# Analisis Kesenjangan Keterampilan (Skill Gap Analysis)

Notebook ini menganalisis kesenjangan antara keahlian yang diajarkan di kurikulum akademik (berdasarkan standar O*NET) dengan keahlian yang dibutuhkan oleh industri (berdasarkan data lowongan kerja Adzuna).

**Alur:** Baca data → Standarisasi nama skill → Hitung frekuensi akademik → Hitung frekuensi industri → Hitung skor gap → Simpan hasil

**Rumus Skor Gap:**

$Gap = Industry_{norm} - Academic_{norm}$

- Skor **positif besar** = skill sangat dicari industri tapi jarang diajarkan (kesenjangan tertinggi)
- Skor **negatif** = skill lebih banyak diajarkan daripada dicari industri

## Langkah 1: Import Library

In [1]:
# ==============================================================================
# LANGKAH 1: Import Library
# Tujuan: Memuat semua pustaka yang dibutuhkan untuk analisis
# ==============================================================================

# Mengimpor pandas untuk manipulasi data tabel (DataFrame)
import pandas as pd

# Mengimpor os untuk memanipulasi path file
import os

# Mengimpor re untuk pencocokan teks skill menggunakan regex
import re

print("Library berhasil dimuat!")

Library berhasil dimuat!


## Langkah 2: Baca Semua Dataset
Kita membaca 3 sumber data:
1. **`adzuna_jobs.csv`** — data lowongan kerja dari Adzuna API (representasi kebutuhan industri)
2. **`Technology Skills.xlsx`** — daftar skill teknologi dari database O*NET (representasi kurikulum akademik)
3. **`Skills.xlsx`** — daftar skill dasar/core dari database O*NET

In [2]:
# ==============================================================================
# LANGKAH 2: Baca Semua Dataset
# Tujuan: Memuat data mentah dari file lokal
# ==============================================================================

# Data lowongan kerja industri (Adzuna API)
df_adzuna = pd.read_csv("adzuna_jobs.csv")
print(f"Data lowongan Adzuna: {len(df_adzuna)} baris")

# Data skill teknologi dari O*NET (seluruhnya dikategorikan Hard Skill)
df_tech_skills = pd.read_excel("db_30_2_excel/Technology Skills.xlsx")
print(f"Data Technology Skills O*NET: {len(df_tech_skills)} baris")

# Data skill dasar/core dari O*NET (berisi campuran Hard dan Soft Skill)
df_core_skills = pd.read_excel("db_30_2_excel/Skills.xlsx")
print(f"Data Core Skills O*NET: {len(df_core_skills)} baris")

Data lowongan Adzuna: 280 baris


Data Technology Skills O*NET: 32773 baris


Data Core Skills O*NET: 62580 baris


## Langkah 3: Fungsi Standarisasi Nama Skill
Nama skill perlu distandardisasi agar pencocokan konsisten — misalnya `"Node.JS"`, `"node js"`, dan `"nodejs"` harus dianggap sama.

In [3]:
# ==============================================================================
# LANGKAH 3: Fungsi Standarisasi Nama Skill
# Tujuan: Menyeragamkan penulisan nama skill agar pencocokan konsisten
# ==============================================================================

def standarkan_nama_skill(teks):
    """
    Membersihkan dan menstandardisasi nama skill.

    Parameter:
        teks (str) -- nama skill mentah yang akan distandardisasi

    Return:
        str -- nama skill yang sudah bersih dan seragam (huruf kecil)
    """
    if pd.isna(teks):
        return teks

    teks = str(teks).lower().strip()

    # Kamus standarisasi singkatan teknologi populer
    standarisasi = {
        r'\bjs\b': 'javascript',
        r'\bnode\.js\b': 'nodejs',
        r'\bnode js\b': 'nodejs',
        r'\bvue\.js\b': 'vuejs',
        r'\breact\.js\b': 'reactjs',
        r'\bml\b': 'machine learning',
        r'\bai\b': 'artificial intelligence',
        r'\bpython3\b': 'python',
        r'\baws\b': 'amazon web services',
    }

    for pola, pengganti in standarisasi.items():
        teks = re.sub(pola, pengganti, teks)

    return teks

# Contoh
print(standarkan_nama_skill("Node.JS"))  # -> nodejs
print(standarkan_nama_skill("AWS"))      # -> amazon web services

node.javascript
amazon web services


## Langkah 4: Fungsi Kategori Skill (Hard / Soft)
Satu fungsi untuk mengklasifikasikan apakah suatu skill adalah Hard Skill atau Soft Skill. Fungsi ini dipakai untuk **kedua** dataset O*NET (Technology Skills dan Core Skills).

In [4]:
# ==============================================================================
# LANGKAH 4: Fungsi Kategori Skill (Hard / Soft)
# Tujuan: Mengklasifikasikan setiap skill ke Hard Skill atau Soft Skill
# ==============================================================================

# Daftar soft skill acuan berdasarkan 35 core skill O*NET
DAFTAR_SOFT_SKILL = {
    'active listening', 'writing', 'speaking', 'social perceptiveness',
    'coordination', 'persuasion', 'negotiation', 'instructing',
    'service orientation', 'complex problem solving', 'critical thinking',
    'active learning', 'learning strategies', 'monitoring',
    'judgment and decision making', 'time management',
    'management of personnel resources', 'management of financial resources',
    'management of material resources',
}


def tentukan_kategori_skill(nama_skill):
    """
    Menentukan apakah suatu skill termasuk Hard Skill atau Soft Skill.

    Parameter:
        nama_skill (str) -- nama skill yang sudah distandardisasi (huruf kecil)

    Return:
        str -- 'Soft Skill' jika nama_skill ada di daftar soft skill, 
               'Hard Skill' jika tidak
    """
    if nama_skill in DAFTAR_SOFT_SKILL:
        return 'Soft Skill'
    return 'Hard Skill'


# Contoh
print(tentukan_kategori_skill('python'))            # -> Hard Skill
print(tentukan_kategori_skill('critical thinking')) # -> Soft Skill

Hard Skill
Soft Skill


## Langkah 5: Hitung Frekuensi Skill di Data Akademik (O*NET)
Menghitung berapa kali setiap skill muncul di kedua dataset O*NET, lalu menggabungkannya menjadi satu DataFrame `df_akademik`.

In [5]:
# ==============================================================================
# LANGKAH 5: Hitung Frekuensi Skill di Data Akademik (O*NET)
# Tujuan: Mengagregasi frekuensi kemunculan setiap skill di kurikulum akademik
# ==============================================================================

# --- 5a: Technology Skills (semua Hard Skill) ---
# Tentukan kolom yang berisi nama skill
kolom_skill_tech = 'Example' if 'Example' in df_tech_skills.columns else df_tech_skills.columns[0]
df_tech_skills['nama_skill'] = df_tech_skills[kolom_skill_tech].apply(standarkan_nama_skill)

# Hitung frekuensi kemunculan per skill
df_tech_freq = df_tech_skills.groupby('nama_skill').size().reset_index(name='freq_academic')
df_tech_freq['kategori'] = 'Hard Skill'

# --- 5b: Core Skills (campuran Hard dan Soft) ---
df_core_skills['nama_skill'] = df_core_skills['Element Name'].apply(standarkan_nama_skill)

# Hitung frekuensi kemunculan per skill
df_core_freq = df_core_skills.groupby('nama_skill').size().reset_index(name='freq_academic')

# Tentukan kategori menggunakan fungsi tentukan_kategori_skill
df_core_freq['kategori'] = df_core_freq['nama_skill'].apply(tentukan_kategori_skill)

# --- 5c: Gabungkan kedua dataset akademik ---
df_akademik_gabungan = pd.concat([df_tech_freq, df_core_freq], ignore_index=True)

# Jika ada skill yang muncul di kedua dataset, jumlahkan frekuensinya
# dan prioritaskan kategori Soft Skill jika salah satu menandainya
def gabung_kategori(daftar_kategori):
    """Jika ada 'Soft Skill' di antara kategori, kembalikan 'Soft Skill'."""
    if 'Soft Skill' in list(daftar_kategori):
        return 'Soft Skill'
    return 'Hard Skill'

df_akademik = df_akademik_gabungan.groupby('nama_skill').agg({
    'freq_academic': 'sum',
    'kategori': gabung_kategori
}).reset_index()

print(f"Total skill unik dari akademik: {len(df_akademik)}")
print(f"  - Hard Skill: {len(df_akademik[df_akademik['kategori'] == 'Hard Skill'])}")
print(f"  - Soft Skill: {len(df_akademik[df_akademik['kategori'] == 'Soft Skill'])}")

Total skill unik dari akademik: 8820
  - Hard Skill: 8801
  - Soft Skill: 19


## Langkah 6: Fungsi Cek Skill di Teks
Fungsi sederhana untuk mengecek apakah suatu nama skill muncul di dalam teks deskripsi lowongan. Menggunakan regex `\b` (word boundary) agar pencocokan akurat — misalnya skill `"r"` tidak salah cocok dengan kata `"react"`.

In [6]:
# ==============================================================================
# LANGKAH 6: Fungsi Cek Skill di Teks
# Tujuan: Mengecek apakah suatu skill muncul di teks deskripsi lowongan
# ==============================================================================

def skill_muncul_di_teks(nama_skill, teks):
    """
    Mengecek apakah nama skill muncul di dalam teks menggunakan regex.

    Parameter:
        nama_skill (str) -- nama skill yang dicari (huruf kecil)
        teks (str) -- teks deskripsi lowongan (huruf kecil)

    Return:
        bool -- True jika skill ditemukan, False jika tidak
    """
    # re.escape() mengamankan karakter khusus (misal '+' di 'c++')
    # \b adalah word boundary — memastikan pencocokan per kata utuh
    pola = r'\b' + re.escape(nama_skill) + r'\b'
    return bool(re.search(pola, teks))


# Contoh
teks_contoh = "looking for python and machine learning experience"
print(skill_muncul_di_teks('python', teks_contoh))           # True
print(skill_muncul_di_teks('java', teks_contoh))             # False
print(skill_muncul_di_teks('machine learning', teks_contoh)) # True

True
False
True


## Langkah 7: Hitung Frekuensi Skill Akademik di Lowongan Industri
Untuk setiap skill yang ada di daftar akademik, kita hitung berapa banyak lowongan Adzuna yang menyebutkan skill tersebut di deskripsinya.

> **Catatan simplifikasi:** Kita menggunakan regex sederhana per skill tanpa optimasi. Untuk skala data ini (~500 lowongan × ~200 skill), hasilnya sudah cukup cepat.

In [7]:
# ==============================================================================
# LANGKAH 7: Hitung Frekuensi Skill Akademik di Lowongan Industri
# Tujuan: Menghitung seberapa sering setiap skill akademik disebutkan
#         di dalam deskripsi lowongan kerja (data industri)
# ==============================================================================

# Ambil semua nama skill unik dari dataset akademik
daftar_skill_akademik = df_akademik['nama_skill'].dropna().unique()

# Siapkan deskripsi lowongan dalam bentuk list (sudah lowercase)
deskripsi_lowongan = df_adzuna['description'].dropna().astype(str).str.lower().tolist()
print(f"Mencocokkan {len(daftar_skill_akademik)} skill terhadap {len(deskripsi_lowongan)} lowongan...")

# Hitung frekuensi kemunculan setiap skill di lowongan industri
frekuensi_industri = {}

for skill in daftar_skill_akademik:
    if not skill:
        continue
    jumlah = 0
    for deskripsi in deskripsi_lowongan:
        if skill_muncul_di_teks(skill, deskripsi):
            jumlah += 1
    frekuensi_industri[skill] = jumlah

# Ubah ke DataFrame
df_industri = pd.DataFrame({
    'nama_skill': list(frekuensi_industri.keys()),
    'freq_industry': list(frekuensi_industri.values())
})

print(f"Selesai! Skill dengan frekuensi industri > 0: {len(df_industri[df_industri['freq_industry'] > 0])}")

Mencocokkan 8820 skill terhadap 280 lowongan...


Selesai! Skill dengan frekuensi industri > 0: 28


## Langkah 8: Fungsi Hitung Skor Gap
Menghitung skor kesenjangan berdasarkan proporsi (normalisasi frekuensi). Rumus:

$Industry_{norm} = \frac{freq\_industry_i}{\sum freq\_industry}$

$Academic_{norm} = \frac{freq\_academic_i}{\sum freq\_academic}$

$Skill\ Gap\ Score = Industry_{norm} - Academic_{norm}$

In [8]:
# ==============================================================================
# LANGKAH 8: Fungsi Hitung Skor Gap
# Tujuan: Menghitung skor kesenjangan berdasarkan proporsi normalisasi
# ==============================================================================

def hitung_skor_gap(df_gabungan, kolom_industri, kolom_akademik):
    """
    Menghitung Skill Gap Score berdasarkan normalisasi frekuensi.

    Parameter:
        df_gabungan (DataFrame) -- DataFrame gabungan frekuensi industri dan akademik
        kolom_industri (str) -- nama kolom yang berisi frekuensi industri
        kolom_akademik (str) -- nama kolom yang berisi frekuensi akademik

    Return:
        DataFrame -- dengan kolom tambahan: industry_norm, academic_norm, skill_gap_score
    """
    df = df_gabungan.copy()

    # Isi nilai kosong dengan 0
    df[kolom_industri] = df[kolom_industri].fillna(0)
    df[kolom_akademik] = df[kolom_akademik].fillna(0)

    # Hitung total frekuensi (untuk normalisasi)
    total_industri = df[kolom_industri].sum()
    total_akademik = df[kolom_akademik].sum()

    # Cegah pembagian dengan nol
    total_industri = total_industri if total_industri > 0 else 1
    total_akademik = total_akademik if total_akademik > 0 else 1

    # Normalisasi: ubah frekuensi absolut menjadi proporsi relatif (0.0 - 1.0)
    # Proporsi menunjukkan seberapa besar kontribusi skill tersebut
    # dibanding seluruh skill di dataset
    df['industry_norm'] = df[kolom_industri] / total_industri
    df['academic_norm'] = df[kolom_akademik] / total_akademik

    # Skor Gap = proporsi industri - proporsi akademik
    # Positif besar = dicari industri tapi jarang diajarkan (kesenjangan tinggi)
    # Negatif = lebih banyak diajarkan daripada dicari
    df['skill_gap_score'] = df['industry_norm'] - df['academic_norm']

    # Urutkan dari kesenjangan tertinggi ke terendah
    df = df.sort_values(by='skill_gap_score', ascending=False).reset_index(drop=True)

    return df

print("Fungsi hitung_skor_gap() siap digunakan.")

Fungsi hitung_skor_gap() siap digunakan.


## Langkah 9: Gabungkan Data dan Hitung Skor Gap
Menggabungkan data frekuensi akademik dan industri, lalu menghitung skor gap untuk setiap skill.

In [9]:
# ==============================================================================
# LANGKAH 9: Gabungkan Data dan Hitung Skor Gap
# Tujuan: Menggabungkan frekuensi akademik dan industri, lalu hitung skor gap
# ==============================================================================

# Gabungkan data industri dan akademik berdasarkan nama_skill
# how='outer' mempertahankan semua skill dari kedua sumber
df_gabungan = pd.merge(df_industri, df_akademik, on='nama_skill', how='outer')

# Isi kategori yang kosong dengan 'Hard Skill' (default)
df_gabungan['kategori'] = df_gabungan['kategori'].fillna('Hard Skill')

print(f"Total skill unik setelah penggabungan: {len(df_gabungan)}")

# Hitung skor gap
df_hasil = hitung_skor_gap(df_gabungan, 'freq_industry', 'freq_academic')

# Tampilkan top 10 skill dengan kesenjangan tertinggi
print("\nTop 10 Skill dengan Kesenjangan Tertinggi:")
print("(Tinggi di industri, rendah di akademik)")
print(df_hasil[['nama_skill', 'kategori', 'industry_norm', 'academic_norm', 'skill_gap_score']].head(10).to_string(index=False))

Total skill unik setelah penggabungan: 8820

Top 10 Skill dengan Kesenjangan Tertinggi:
(Tinggi di industri, rendah di akademik)
  nama_skill   kategori  industry_norm  academic_norm  skill_gap_score
installation Hard Skill       0.202020       0.018751         0.183269
     analyze Hard Skill       0.131313       0.000010         0.131303
      python Hard Skill       0.131313       0.001384         0.129929
   snowflake Hard Skill       0.070707       0.000052         0.070655
           r Hard Skill       0.050505       0.001070         0.049435
     science Hard Skill       0.060606       0.018751         0.041855
     tableau Hard Skill       0.040404       0.000766         0.039638
coordination Soft Skill       0.050505       0.018751         0.031754
     pyspark Hard Skill       0.020202       0.000021         0.020181
      google Hard Skill       0.020202       0.000042         0.020160


## Langkah 10: Simpan Hasil ke CSV
Menyimpan hasil analisis ke file `skill_gap_analysis.csv` yang akan dibaca oleh dashboard `app.py`.

In [10]:
# ==============================================================================
# LANGKAH 10: Simpan Hasil ke CSV
# Tujuan: Menyimpan hasil analisis agar bisa dibaca oleh dashboard app.py
# ==============================================================================

# Pilih kolom yang relevan untuk disimpan
kolom_simpan = ['nama_skill', 'kategori', 'freq_industry', 'freq_academic',
                'industry_norm', 'academic_norm', 'skill_gap_score']
kolom_tersedia = [k for k in kolom_simpan if k in df_hasil.columns]

df_hasil[kolom_tersedia].to_csv('skill_gap_analysis.csv', index=False)

print(f"Hasil analisis disimpan ke: skill_gap_analysis.csv")
print(f"Total baris: {len(df_hasil)}")
print(f"\nFile ini akan dibaca oleh dashboard app.py untuk visualisasi.")

Hasil analisis disimpan ke: skill_gap_analysis.csv
Total baris: 8820

File ini akan dibaca oleh dashboard app.py untuk visualisasi.
